# PSMO Oncologists — Bronze Ingestion

Ingests the Philippine Society of Medical Oncologists (PSMO) oncologist count by region/province from the landing zone CSV into the bronze layer.

In [ ]:
import os
from datetime import datetime
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# --- Widgets (parameterize for job runs) ---
dbutils.widgets.text("as_of_date", "2026-06-30", "As Of Date")
dbutils.widgets.text("source_file_path", "/Volumes/workspace/default/ftw-b12-de/groups/g5_landing/workforce/psmo_oncologists/psmo_oncologists.csv", "Source File Path")

as_of_date_str = dbutils.widgets.get("as_of_date")
source_path = dbutils.widgets.get("source_file_path")

# Validate as_of_date format
try:
    datetime.strptime(as_of_date_str, "%Y-%m-%d")
except ValueError:
    raise ValueError(f"Invalid as_of_date '{as_of_date_str}'. Expected format: YYYY-MM-DD")

# Explicit schema — avoids extra scan and enforces correct types
schema = StructType([
    StructField("region", StringType(), True),
    StructField("province_area", StringType(), True),
    StructField("doctor_count", IntegerType(), True),
])

# Read the CSV with explicit schema
df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(source_path)
)

# Add metadata columns
source_file = os.path.basename(source_path)
df = df.withColumn("source_file", lit(source_file)) \
       .withColumn("as_of_date", lit(as_of_date_str).cast("date")) \
       .withColumn("ingested_at", current_timestamp())

display(df.limit(10))

In [ ]:
from delta.tables import DeltaTable

# Target bronze table
table_name = "public_health.bronze.psmo_oncologists"

# Create the table if it doesn't exist (with Delta column mapping per project convention)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {table_name} (
    region STRING,
    province_area STRING,
    doctor_count INT,
    source_file STRING,
    as_of_date DATE,
    ingested_at TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'minReaderVersion' = '2',
    'minWriterVersion' = '5'
)
""")

# Idempotent MERGE: upsert on (region, province_area, as_of_date)
# Re-running with the same as_of_date replaces only matching rows;
# data from other snapshot dates is preserved.
merge_condition = (
    "t.region = s.region "
    "AND t.province_area = s.province_area "
    "AND t.as_of_date = s.as_of_date"
)

source_count = df.count()

delta_table = DeltaTable.forName(spark, table_name)

delta_table.alias("t").merge(
    df.alias("s"), merge_condition
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

final_count = spark.table(table_name).count()
print(f"Merged {source_count} source rows into {table_name} (total rows now: {final_count})")